# Task 4 – Supervised Classification / Regression Modeling & Tuning
This notebook trains and compares multiple ML models, performs hyperparameter tuning, evaluates metrics, and saves the best model.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import joblib


In [2]:
data = pd.read_csv('preprocessed_data.csv')  # replace with your actual file name
X = data.drop('target', axis=1)
y = data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


FileNotFoundError: [Errno 2] No such file or directory: 'preprocessed_data.csv'

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    'LightGBM': LGBMClassifier()
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'Report': classification_report(y_test, y_pred, output_dict=True),
        'ROC-AUC': roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    }

pd.DataFrame(results).T


In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=5, scoring='f1')
grid.fit(X_train, y_train)
print("Best Parameters:", grid.best_params_)


In [ ]:
best_model = grid.best_estimator_
y_pred_proba = best_model.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

plt.plot(fpr, tpr, label='ROC curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


In [ ]:
joblib.dump(best_model, 'champion_model.pkl')


## Summary
- Trained 4 models: Logistic Regression, Random Forest, XGBoost, LightGBM  
- Tuned hyperparameters using GridSearchCV  
- Evaluated with Precision, Recall, F1, ROC‑AUC  
- Saved the best model as `champion_model.pkl`
